# CP4 - Mode 2 retrieval on D4 (Colab run)

This notebook runs the CP4 retrieval evaluation with **real, free, open models** on a free Colab GPU and
produces a small zip of result JSON files to hand to the next Claude Code session.

What it does, in order: clone the repository, install Go and the project with its `[ml]` extra, rebuild the D4
evidence (`dataset materialize`, on the local Colab disk), **stop if that evidence does not match the committed case
list**, embed every chunk (cached on Drive), run every retrieval configuration, and zip `cases.json` + `runs/*.json`.

**Zero cost:** only open models from Hugging Face run locally on the Colab machine. No paid API is called.
**What the zip contains:** result JSON only - no repository code, no model files, no embeddings, no source code from
the analysed repositories.

Set `Runtime -> Change runtime type -> T4 GPU` before running. Without a GPU the notebook still runs the small
configurations on CPU (one embedder, no reranker). Run the cells top to bottom; every step can be re-run safely.

**Running time per step:** not measured yet - the cells below say what the Claude Code session observed on CPU
where it could; please note the GPU times you see, they go into the CP4 report.

**Where files live (fix round FIX-4):** git clones, the vulndb snapshot and the rebuilt evidence live on the local Colab
disk (`/content/cache`, `/content/data`): git misbehaves on the Google Drive mount, which is the suspected cause of the
first invalid run. Only embeddings, the materialise summary and the final result JSON go on Drive. After a disconnect the
evidence is rebuilt (about 25 minutes); embeddings are not.


In [ ]:
# Settings for this run. BRANCH is the branch holding the CP4 code; DRY_RUN=True swaps every real model for the
# deterministic fakes (a quick end-to-end check that needs no model download).
BRANCH = "claude/vulntrace-ai-build-9qhbjg"
REPO_URL = "https://github.com/AMAN-PRIYANSH/REPO-MIND.git"
DRY_RUN = False
GO_VERSION = "1.24.7"

In [ ]:
# Check whether a GPU is attached. The embedding and reranking steps use it; everything else runs on the CPU.
import shutil, subprocess
GPU = shutil.which("nvidia-smi") is not None and subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
print("GPU available:", GPU)
if GPU:
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
DEVICE = "cuda" if GPU else "cpu"

In [ ]:
# Mount Google Drive for the files worth keeping across a Colab disconnect: embeddings, the materialise summary and the
# result JSON. The git cache and the rebuilt evidence stay on the LOCAL disk (git behaves badly on the Drive mount).
import os
from google.colab import drive
drive.mount("/content/drive")
WORK = "/content/drive/MyDrive/vulntrace_cp4"      # Drive: embeddings, summaries, results
os.makedirs(WORK, exist_ok=True)
CACHE = "/content/cache"                           # local disk: git clones + vulndb snapshot
DATA = "/content/data"                             # local disk: rebuilt D4 evidence
os.makedirs(CACHE, exist_ok=True)
os.environ["VULNTRACE_CACHE"] = CACHE
print("Drive work directory:", WORK, "| local git cache:", CACHE, "| local evidence:", DATA)


In [ ]:
# Clone (or update) the repository at the CP4 branch. The code that does the work lives in the repository; this
# notebook only calls it.
REPO = "/content/REPO-MIND"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO, "checkout", "-B", BRANCH, f"origin/{BRANCH}"], check=True)
print(subprocess.run(["git", "-C", REPO, "log", "--oneline", "-1"], capture_output=True, text=True).stdout)
%cd /content/REPO-MIND

In [ ]:
# Install Go (Colab does not ship it). The project's small Go helper parses Go files to find function boundaries
# while the D4 evidence is rebuilt. Installed under /content only, nothing system-wide.
if not os.path.exists(f"/content/go{GO_VERSION}/go/bin/go"):
    subprocess.run(["wget", "-q", f"https://go.dev/dl/go{GO_VERSION}.linux-amd64.tar.gz", "-O", "/content/go.tgz"], check=True)
    os.makedirs(f"/content/go{GO_VERSION}", exist_ok=True)
    subprocess.run(["tar", "-C", f"/content/go{GO_VERSION}", "-xzf", "/content/go.tgz"], check=True)
os.environ["PATH"] = f"/content/go{GO_VERSION}/go/bin:" + os.environ["PATH"]
os.environ["GOTOOLCHAIN"] = "local"
os.environ["CGO_ENABLED"] = "0"
print(subprocess.run(["go", "version"], capture_output=True, text=True).stdout)

In [ ]:
# Install this project with the [ml] extra (sentence-transformers, which brings the open embedding and reranker
# models' code). Colab already has torch with CUDA, so this is quick.
!pip install -q -e ".[ml]"
!python -m vulntrace doctor

In [ ]:
# Fetch the pinned golang/vulndb snapshot and the Go helper's dependencies (both cached; quick on re-runs).
!python -m vulntrace dataset fetch
!python -m vulntrace helper fetch
!python -m vulntrace helper build

In [ ]:
# Rebuild the D4 evidence for exactly the 120 committed case IDs (it never re-selects cases) on the local disk. This is the
# slow, network-heavy step: sequential git fetches with a pause between them. A finished case is skipped on a re-run only if
# its stored result matches docs/results/datasets/D4_cases.json; a failed or mismatching one is rebuilt. The last lines name
# the most common git errors, so a git problem is shown as git reported it. Expect about 25 minutes on a 4-core machine.
!python -u -m vulntrace dataset materialize --set d4 --data-dir "{DATA}" --summary-out "{WORK}/D4_materialize.json"


In [ ]:
# GATE: stop here if the rebuilt evidence does not match the committed case list (more than 3 cases with a different
# status, or no candidate functions at all). The first Colab run went on regardless and produced empty results. If this
# cell stops the notebook, do NOT continue: read the git errors printed by the cell above, or run
#   !python -m vulntrace dataset diagnose --case GO-2020-0038
import json
from vulntrace.eval import evidence_gate
case_list = json.load(open("docs/results/datasets/D4_cases.json"))
statuses, n_candidates = evidence_gate.evidence_statuses(case_list, DATA)
problems = evidence_gate.check_evidence(case_list, statuses, n_candidates)
from collections import Counter
print("materialised statuses:", dict(Counter(statuses.values())), "| expected:", case_list.get("status_counts"))
print("total candidate functions:", n_candidates)
if problems:
    raise SystemExit("STOP - the evidence is not usable: " + "; ".join(problems))
print("gate passed")


In [ ]:
# Check that each real model obeys the same interface rules the tests check on the fakes (vectors normalised,
# identical texts give identical vectors, ...). A model that fails here is reported, not silently replaced.
from vulntrace.eval import cp4_plan
from vulntrace.models import contract
from vulntrace.models.registry import make_embedder, make_reranker
model_problems = {}
if not DRY_RUN:
    for name in (cp4_plan.EMBEDDERS if GPU else cp4_plan.EMBEDDERS[:1]):
        try:
            contract.check_embedder(make_embedder(name, device=DEVICE)); print("embedder ok:", name)
        except Exception as exc:
            model_problems[name] = repr(exc); print("EMBEDDER FAILED:", name, exc)
    for name in (cp4_plan.RERANKERS if GPU else ()):
        try:
            contract.check_reranker(make_reranker(name, device=DEVICE)); print("reranker ok:", name)
        except Exception as exc:
            model_problems[name] = repr(exc); print("RERANKER FAILED:", name, exc)
print("problems:", model_problems or "none")

In [ ]:
# Embed every chunk of every release range once per model and chunk variant, caching vectors on Drive by the
# chunk text's hash (re-running skips everything already embedded). This is where the GPU matters: the two
# largest D4 ranges have about 14,000 and 15,000 candidate functions each.
emb = [] if DRY_RUN else [e for e in (cp4_plan.EMBEDDERS if GPU else cp4_plan.EMBEDDERS[:1]) if e not in model_problems]
for e in emb:
    !python -u -m vulntrace index build --set d4 --embedder {e} --data-dir "{DATA}" --embedding-cache "{WORK}/embeddings" --device {DEVICE}

In [ ]:
# Run every configuration: baselines (random, size, B-ID), BM25, dense, hybrid (RRF) and hybrid+rerank, each on
# the three chunk variants. Results go to WORK/results/cp4 as small JSON files; one call per group so each
# embedder's vectors are freed before the next one is loaded. Configurations whose model failed above are skipped.
OUT = f"{WORK}/results/cp4" + ("_dryrun" if DRY_RUN else "")
for group in cp4_plan.configs(gpu=GPU, dry_run=DRY_RUN):
    group = [c for c in group if not any(p in c for p in model_problems)]
    if not group:
        continue
    args = " ".join(f"--config {c}" for c in group)
    print(">>>", args)
    !python -u -m vulntrace eval retrieval {args} --data-dir "{DATA}" --embedding-cache "{WORK}/embeddings" --device {DEVICE} --out "{OUT}"

In [ ]:
# Render the results document from the files just written, to look at it here. The next Claude Code session
# renders the committed copy the same way from the zip.
!python -m vulntrace results render --doc cp4 --input "{OUT}" --out "{WORK}/CP4_RETRIEVAL_RESULTS_preview.md"
from IPython.display import Markdown, display
display(Markdown(open(f"{WORK}/CP4_RETRIEVAL_RESULTS_preview.md").read()[:6000]))

In [ ]:
# Zip only the result JSON (cases.json, runs/*.json, the materialisation summary and any model problems) and
# download it. No code, no models, no embeddings, no analysed source code go into the zip.
import json, zipfile
json.dump(model_problems, open(f"{OUT}/model_problems.json", "w"), indent=1)
zpath = f"{WORK}/cp4_results{'_dryrun' if DRY_RUN else ''}.zip"
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(f"{OUT}/cases.json", "cp4/cases.json")
    z.write(f"{OUT}/model_problems.json", "cp4/model_problems.json")
    z.write(f"{WORK}/D4_materialize.json", "cp4/D4_materialize_colab.json")
    for name in sorted(os.listdir(f"{OUT}/runs")):
        z.write(f"{OUT}/runs/{name}", f"cp4/runs/{name}")
print(zpath, os.path.getsize(zpath) // 1024, "KiB")
from google.colab import files
files.download(zpath)